In [7]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [8]:
from rag_helper import RAGbase
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client=OpenAI()

In [3]:
index.search("How to run other llms?",num_results=1)

[{'course': 'machine-learning-zoomcamp',
  'section': 'Module 1. Introduction to Machine Learning',
  'question': 'How to select column by dtype',
  'answer': "To select columns by data type, you can use the following methods:\n\n- To get columns with numeric data:\n\n  ```python\n  df.select_dtypes(include=np.number).columns.tolist()\n  ```\n  \n- To get columns with object (string) data:\n\n  ```python\n  df.select_dtypes(include='object').columns.tolist()\n  ```",
  'doc_id': 'eecebb4f2f'}]

In [9]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [10]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [11]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [12]:
response=openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [15]:
call=response.output[0]
call

ResponseFunctionToolCall(arguments='{"query":"join course discovered can I join enrollment late registration eligibility"}', call_id='call_ZZBc1ACWnRX6WVtceMkvUliV', name='search', type='function_call', id='fc_079f37a5e11a09d1006a398f84f38481a09efab6d803d0e084', namespace=None, status='completed')

In [16]:
import json

args=json.loads(call.arguments)

In [18]:
args

{'query': 'join course discovered can I join enrollment late registration eligibility'}

In [17]:
call.name

'search'

In [20]:
results = search(**args)
results_json = json.dumps(results, indent=2)

In [24]:
function_call_output={
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": results_json,
}

In [25]:
messages.append(call)
messages.append(function_call_output)

In [27]:
response=openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [28]:
response.output_text

'Yes — you can still join the course.\n\nIf your goal is just to learn, you can start anytime. If you want a certificate, you need to submit your project while submissions are still open.'

In [31]:
response.usage.input_tokens, response.usage.output_tokens

(776, 44)

In [32]:
response.output

[ResponseOutputMessage(id='msg_079f37a5e11a09d1006a3995cd012c81a08258912ea3e0145b', content=[ResponseOutputText(annotations=[], text='Yes — you can still join the course.\n\nIf your goal is just to learn, you can start anytime. If you want a certificate, you need to submit your project while submissions are still open.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]